# Targeted Carlini-Wagner (CW) Attack

This notebook demonstrates the targeted Carlini-Wagner attack on Whisper.

**Goal:** Modify an audio sample so Whisper outputs a *specific target phrase* instead of the original content.

**Method:**
1.  Initialize perturbation (random noise).
2.  Optimize perturbation to minimize $L_2$ distance from original audio.
3.  Optimize perturbation to *maximize* the log-probability of the target tokens.
4.  Apply constraints.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import torch
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from tqdm import tqdm
import random
import pathlib
import pandas as pd

from IPython.display import Audio, display
import jiwer

import src.data as data_loader
from src.data import get_librispeech_files, load_audio_tensor, DATA_DIR
import src.attacks as attacks
from src.attacks import CWAuditoryAttack, compute_snr
from src.models.whisper_wrapper import WhisperASRWithAttack

torch.manual_seed(42)
np.random.seed(42)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")


In [ ]:
# Load Model (consistent with notebooks 03 & 04)
model = WhisperASRWithAttack(model_path="openai/whisper-base", device=device)
print("Model loaded.")


In [ ]:
# Load a sample utterance from LibriSpeech test-clean
all_files = get_librispeech_files()
assert len(all_files) > 0, "No audio files found — check DATA_DIR"

# Pick a random file reproducibly
random.seed(42)
audio_path = random.choice(all_files)
audio_path = pathlib.Path(audio_path)

# Read the ground-truth transcript from the accompanying .trans.txt file
trans_file = audio_path.parent / f"{audio_path.parent.parent.name}-{audio_path.parent.name}.trans.txt"
ground_truth = ""
if trans_file.exists():
    with open(trans_file) as f:
        for line in f:
            utt_id, *words = line.strip().split()
            if utt_id == audio_path.stem:
                ground_truth = " ".join(words).lower()
                break

_, clean_audio = load_audio_tensor(str(audio_path))   # float32 tensor in [-1, 1]

print(f"Audio path : {audio_path}")
print(f"Duration   : {clean_audio.shape[0] / 16000:.2f}s  ({clean_audio.shape[0]} samples)")
print(f"Ground truth: {ground_truth}")
display(Audio(clean_audio.numpy(), rate=16000))


In [ ]:
# Define the target phrase Whisper should output after the attack
target_phrase = "hello world"
print(f"Target phrase : {target_phrase}")

# Instantiate CW attack
cw_attack = CWAuditoryAttack(
    whisper_model=model,
    device=device,
    learning_rate=0.01,
    c=1.0,
    steps=50,
)

# Run attack — returns 1-D adversarial tensor on CPU
adv_audio = cw_attack.attack(clean_audio, target_phrase)

# Save both versions for playback / evaluation
sf.write("targeted_attack_clean.wav", clean_audio.numpy(), samplerate=16000)
sf.write("targeted_attack_adv.wav",   adv_audio.numpy(),   samplerate=16000)

snr = compute_snr(clean_audio.numpy(), adv_audio.numpy())
print(f"\nSNR (clean vs adversarial): {snr:.2f} dB")
print("Files saved.")


In [ ]:
# Evaluate Results
print("=" * 55)
print("  EVALUATION")
print("=" * 55)

pred_clean = model.transcribe(clean_audio)
pred_adv   = model.transcribe(adv_audio)

wer_clean = jiwer.wer(ground_truth, pred_clean.lower())
wer_adv   = jiwer.wer(ground_truth, pred_adv.lower())

attack_success = target_phrase.lower() in pred_adv.lower()

print(f"Ground truth    : {ground_truth}")
print(f"Target phrase   : {target_phrase}")
print()
print(f"Clean prediction: {pred_clean}")
print(f"Adv. prediction : {pred_adv}")
print()
print(f"WER  (clean)    : {wer_clean:.3f}")
print(f"WER  (adv)      : {wer_adv:.3f}")
print(f"SNR             : {snr:.2f} dB")
print(f"Attack success  : {'YES ✓' if attack_success else 'NO  ✗'}")
print("=" * 55)

# --- Waveform & spectrogram comparison ---
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
t = np.linspace(0, len(clean_audio) / 16000, len(clean_audio))

axes[0, 0].plot(t, clean_audio.numpy(), linewidth=0.4)
axes[0, 0].set_title("Clean waveform")
axes[0, 0].set_xlabel("Time (s)")

axes[0, 1].plot(t[:len(adv_audio)], adv_audio.numpy(), linewidth=0.4, color="orange")
axes[0, 1].set_title("Adversarial waveform")
axes[0, 1].set_xlabel("Time (s)")

delta_np = (adv_audio - clean_audio[:len(adv_audio)]).numpy()
axes[1, 0].plot(t[:len(delta_np)], delta_np, linewidth=0.4, color="red")
axes[1, 0].set_title(f"Perturbation δ  (SNR={snr:.1f} dB)")
axes[1, 0].set_xlabel("Time (s)")

axes[1, 1].specgram(delta_np, Fs=16000, cmap="inferno")
axes[1, 1].set_title("Perturbation spectrogram")
axes[1, 1].set_xlabel("Time (s)")
axes[1, 1].set_ylabel("Freq (Hz)")

plt.tight_layout()
plt.show()

print("\n--- Playback ---")
print("Clean:")
display(Audio(clean_audio.numpy(), rate=16000))
print("Adversarial:")
display(Audio(adv_audio.numpy(), rate=16000))


### Observations
- Targeted attacks are significantly harder than untargeted attacks (which just try to break the model).
- The perturbation might be larger to successfully override the original semantic information.
- If `result_adv['text']` contains the target phrase, the attack is successful.

In [ ]:

# Batch evaluation — 5 utterances, single target phrase
N_SAMPLES  = 5
TARGET     = "hello world"
STEPS      = 50

batch_cw = CWAuditoryAttack(
    whisper_model=model,
    device=device,
    learning_rate=0.01,
    c=1.0,
    steps=STEPS,
)

random.seed(42)
sample_files = random.sample(all_files, min(N_SAMPLES, len(all_files)))

rows = []
for fp in sample_files:
    fp = pathlib.Path(fp)
    _, audio = load_audio_tensor(str(fp))

    # Ground truth
    tf = fp.parent / f"{fp.parent.parent.name}-{fp.parent.name}.trans.txt"
    gt = ""
    if tf.exists():
        with open(tf) as f:
            for line in f:
                uid, *words = line.strip().split()
                if uid == fp.stem:
                    gt = " ".join(words).lower()
                    break

    adv = batch_cw.attack(audio, TARGET)

    pred_c = model.transcribe(audio).lower()
    pred_a = model.transcribe(adv).lower()
    snr_i  = compute_snr(audio.numpy(), adv.numpy())
    succ   = TARGET in pred_a

    rows.append({
        "file"        : fp.name,
        "ground_truth": gt,
        "clean_pred"  : pred_c,
        "adv_pred"    : pred_a,
        "WER_clean"   : round(jiwer.wer(gt, pred_c), 3),
        "WER_adv"     : round(jiwer.wer(gt, pred_a), 3),
        "SNR_dB"      : round(snr_i, 2),
        "success"     : succ,
    })
    print(f"[{'✓' if succ else '✗'}] {fp.name}  SNR={snr_i:.1f}dB  adv='{pred_a}'")

results_df = pd.DataFrame(rows)
print("\n--- Batch Summary ---")
print(results_df[["file", "WER_clean", "WER_adv", "SNR_dB", "success"]].to_string(index=False))
print(f"\nAttack success rate : {results_df['success'].mean():.0%}")
print(f"Mean SNR            : {results_df['SNR_dB'].mean():.2f} dB")
print(f"Mean WER (clean)    : {results_df['WER_clean'].mean():.3f}")
print(f"Mean WER (adv)      : {results_df['WER_adv'].mean():.3f}")
